# Check blocks and ball radii

Rebuilds the PointNet++ hierarchy on sampled training blocks (farthest point sampling with `model.saRatios`) and, for each set abstraction level, counts the points that fall inside the ball of radius `model.saRadii`.

We should aim for 
- median_in_ball around nsample (32) or somewhat above, and pct_below_4 close to zero, 
- otherwise copy suggested_r (the radius at which a typical ball holds exactly nsample points) into model.saRadii in `configs/datasets/{dataset}.yaml`.

The block size and points per block are the RandLA-Net values and should stay as they are, so both reference models see the same blocks. We could try alternative radii through `OVERRIDES`

## Setup

In [ ]:
import os, sys
ROOT = os.path.abspath('..')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

from src import paths
print('project folder:', paths.PROJECT_ROOT)

from src.config import loadConfig
from src.diagnostics import checkBlocks

In [ ]:
DATASET   = 'toronto3d'
N_BLOCKS  = 20
OVERRIDES = [] #like ['model.saRadii=[0.5, 1.2, 2.5, 5.0]']

## Run

In [ ]:
cfg = loadConfig(DATASET, OVERRIDES)
summary, levelTable, blockSizes, counts = checkBlocks(cfg, N_BLOCKS)
for w in summary['warnings']:
    print('WARNING:', w)
display(pd.Series({k: v for k, v in summary.items() if k not in ('inference_chunks', 'warnings')}).to_frame(DATASET))
display(levelTable)
print('Inference chunks per test file:', summary['inference_chunks'])

## Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].hist(blockSizes, bins=40, color='steelblue')
axes[0].axvline(cfg['blocks']['numPoints'], color='red', ls='--', label='numPoints')
axes[0].set_title('points per training block'); axes[0].legend()
axes[1].boxplot([np.maximum(c, 1) for c in counts])
axes[1].set_xticks(range(1, len(counts) + 1)); axes[1].set_xticklabels([f'SA{i + 1}' for i in range(len(counts))])
for i, ns in enumerate(cfg['model']['saNsample']):
    axes[1].hlines(ns, i + 0.6, i + 1.4, color='red', ls='--')
axes[1].set_yscale('log'); axes[1].set_ylabel('points inside the ball')
axes[1].set_title('ball occupancy per level (red: nsample)')
plt.tight_layout()
plt.show()